In [ ]:
# Copyright 2026 Google LLC
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# Search Your Documents with SearchDocuments in Agent Retrieval

This notebook introduces **SearchDocuments**, a single API call that searches the files you ingested into an **[Agent Retrieval](https://docs.cloud.google.com/gemini-enterprise-agent-platform/build/vector-search-2/overview)** (formerly Vector Search 2.0) data store with **managed ingestion**. It is written for developers who are familiar with vector search and embeddings concepts, but new to this Google Cloud service.

**New to Agent Retrieval?** If you want to learn the underlying Collections, Data Objects, and search APIs first, see: [Introduction to Agent Platform Vector Search 2.0](https://github.com/GoogleCloudPlatform/generative-ai/blob/main/embeddings/vector-search-2-intro.ipynb)

<table align="left">
  <td style="text-align: center">
    <a href="https://colab.research.google.com/github/GoogleCloudPlatform/generative-ai/blob/main/embeddings/agent-retrieval-search-documents.ipynb">
      <img width="32px" src="https://www.gstatic.com/pantheon/images/bigquery/welcome_page/colab-logo.svg" alt="Google Colaboratory logo"><br> Open in Colab
    </a>
  </td>
  <td style="text-align: center">
    <a href="https://console.cloud.google.com/agent-platform/colab/import/https:%2F%2Fraw.githubusercontent.com%2FGoogleCloudPlatform%2Fgenerative-ai%2Fmain%2Fembeddings%2Fagent-retrieval-search-documents.ipynb">
      <img width="32px" src="https://lh3.googleusercontent.com/JmcxdQi-qOpctIvWKgPtrzZdJJK-J3sWE1RsfjZNwshCFgE_9fULcNpuXYTilIR2hjwN" alt="Google Cloud Colab Enterprise logo"><br> Open in Colab Enterprise
    </a>
  </td>
  <td style="text-align: center">
    <a href="https://console.cloud.google.com/agent-platform/workbench/instances?download_url=https://raw.githubusercontent.com/GoogleCloudPlatform/generative-ai/main/embeddings/agent-retrieval-search-documents.ipynb">
      <img width="32px" src="https://storage.googleapis.com/github-repo/workbench-icon.svg" alt="Workbench logo"><br> Open in Workbench
    </a>
  </td>
  <td style="text-align: center">
    <a href="https://github.com/GoogleCloudPlatform/generative-ai/blob/main/embeddings/agent-retrieval-search-documents.ipynb">
      <img width="32px" src="https://raw.githubusercontent.com/primer/octicons/refs/heads/main/icons/mark-github-24.svg" alt="GitHub logo"><br> View on GitHub
    </a>
  </td>
</table>

<div style="clear: both;"></div>

<p>
<b>Share to:</b>

<a href="https://www.linkedin.com/sharing/share-offsite/?url=https%3A//github.com/GoogleCloudPlatform/generative-ai/blob/main/embeddings/agent-retrieval-search-documents.ipynb" target="_blank">
  <img width="20px" src="https://upload.wikimedia.org/wikipedia/commons/8/81/LinkedIn_icon.svg" alt="LinkedIn logo">
</a>

<a href="https://bsky.app/intent/compose?text=https%3A//github.com/GoogleCloudPlatform/generative-ai/blob/main/embeddings/agent-retrieval-search-documents.ipynb" target="_blank">
  <img width="20px" src="https://upload.wikimedia.org/wikipedia/commons/7/7a/Bluesky_Logo.svg" alt="Bluesky logo">
</a>

<a href="https://twitter.com/intent/tweet?url=https%3A//github.com/GoogleCloudPlatform/generative-ai/blob/main/embeddings/agent-retrieval-search-documents.ipynb" target="_blank">
  <img width="20px" src="https://upload.wikimedia.org/wikipedia/commons/5/5a/X_icon_2.svg" alt="X logo">
</a>

<a href="https://reddit.com/submit?url=https%3A//github.com/GoogleCloudPlatform/generative-ai/blob/main/embeddings/agent-retrieval-search-documents.ipynb" target="_blank">
  <img width="20px" src="https://redditinc.com/hubfs/Reddit%20Inc/Brand/Reddit_Logo.png" alt="Reddit logo">
</a>

<a href="https://www.facebook.com/sharer/sharer.php?u=https%3A//github.com/GoogleCloudPlatform/generative-ai/blob/main/embeddings/agent-retrieval-search-documents.ipynb" target="_blank">
  <img width="20px" src="https://upload.wikimedia.org/wikipedia/commons/5/51/Facebook_f_logo_%282019%29.svg" alt="Facebook logo">
</a>
</p>

| Author |
| --- |
| [Kaz Sato](https://github.com/kazunori279) |

## What is SearchDocuments?

Building document search on a vector database usually means writing a pipeline: parse the files, split them into chunks, embed each chunk, write the chunks to a collection, then learn that collection's schema to query it. Agent Retrieval offers two features that remove that work:

- **Managed ingestion**: Point a **data store** at files in Cloud Storage (PDF, TXT, DOCX, PPTX, XLSX). The service parses them, splits them into segments, embeds each segment with `gemini-embedding-2`, and writes the segments into an Agent Retrieval Collection that it creates for you.
- **SearchDocuments**: Send a text query to the data store and get back ranked segments with their text, page range, and source file URI. Under the hood it runs **hybrid search** (semantic + full-text) and can optionally **rerank** the results with the Vertex AI ranking API.

### What changes compared to searching the Collection directly

| Without SearchDocuments | With SearchDocuments |
|---|---|
| Read `collectionName` from the data store and pass it as `parent` | Pass the data store itself as `parent` |
| Know the vector field name (`multimodal_embedding`) | Not needed |
| Set `task_type=RETRIEVAL_QUERY` | Not needed |
| Set `output_fields` or get no text back | Snippets, page ranges, and source URIs come back by default |
| Wire hybrid search (semantic + text) yourself with `BatchSearchDataObjects` | Hybrid search runs automatically |
| Call the ranking API yourself | One toggle: `ranking_spec.reranking_spec.enabled` |
| Filter on stored field names (`year`, `google_unstructured_details.start_page`) | Filter on the paths a result carries (`metadata.year`, `unstructured_details.page_range.start_page`) |
| List each metadata field in `output_fields.data_fields` | Name properties in `output_fields.metadata`, or pass `"*"` for all of them |

If you need full control over search (custom batch searches, vector-only search, ANN index hints), you can still query the backing Collection directly with the [search APIs](https://docs.cloud.google.com/gemini-enterprise-agent-platform/build/vector-search-2/query-search/search). SearchDocuments is the simple path for the common case.

For the full API reference of data stores and managed ingestion, see [Search documents in a data store](https://docs.cloud.google.com/gemini-enterprise-agent-platform/build/vector-search-2/query-search/search-data-stores).

### Core Concepts

1. **Data store**: The ingestion resource. It holds your documents and the configuration that says how to parse and segment them. One data store handles one modality (`UNSTRUCTURED` documents or `VIDEO`), fixed at creation.
2. **Document**: One file from Cloud Storage, with optional metadata you attach to it (for example, `year` or `doc_type`).
3. **Segment**: A piece of a document (a chunk or a page) with its own embedding. SearchDocuments returns segments, not whole documents.
4. **Backing Collection**: The Agent Retrieval Collection the data store creates and writes segments into. You don't need it for SearchDocuments, but it is a regular Collection you can also query directly.

> **Note:** The data store and SearchDocuments APIs are served at `v1alpha` and are not yet in the `google-cloud-vectorsearch` Python SDK. This notebook calls them over REST with an authorized session.

Let's explore each step with hands-on examples!

## Example Scenario: Financial Report Search

To demonstrate SearchDocuments, we'll build a **question-answering search over Alphabet's investor reports**: an analyst asks questions in plain language and gets back the exact pages that answer them, with a citation to the source PDF.

**For this demo**: We'll ingest **18 PDFs** from a public Cloud Storage bucket: the 16 quarterly earnings releases from 2021 to 2024, and the 2021 and 2022 annual reports.

### Business Use Cases:

1. **Natural language questions**: "How much did Google Cloud revenue grow?"
2. **Filtered search**: "Only search earnings releases from 2023 onward"
3. **Page-aware search**: "Only search the first page of each release, where the executive quotes are"
4. **Document-scoped search**: "Compare operating income in Q4 2023 and Q4 2024"
5. **Better ranking**: Rerank results so the most relevant pages come first

### Dataset Overview:

Each PDF becomes one document. We attach three metadata properties to each document so we can filter and return them in search:

- `doc_type`: `earnings_release` or `annual_report`
- `year`: Fiscal year (e.g., 2023)
- `quarter`: Fiscal quarter (1-4). Earnings releases only; annual reports don't have this property.

### Sample Data:

| Document ID | File | doc_type | year | quarter |
|---|---|---|---|---|
| 2023-q4-earnings-release | 2023q4-alphabet-earnings-release.pdf | earnings_release | 2023 | 4 |
| 2024-q1-earnings-release | 2024q1-alphabet-earnings-release-pdf.pdf | earnings_release | 2024 | 1 |
| 2022-annual-report | 2022_alphabet_annual_report.pdf | annual_report | 2022 | |

Throughout this notebook, we'll walk through each step: staging the documents, creating a data store, importing the documents, and then searching them with SearchDocuments. Let's get started!

-----

## Prerequisites

This tutorial requires a Google Cloud project that is linked with a billing account. To create a new project, take a look at [this document](https://docs.cloud.google.com/gemini-enterprise-agent-platform/machine-learning/start/cloud-environment) to create a project and setup a billing account for it.
To get the permissions that you need to give a service account access to enable APIs and interact with Agent Platform resources, ask your administrator to grant you the [Security Admin](https://cloud.google.com/iam/docs/roles-permissions/iam#iam.securityAdmin) (`roles/iam.securityAdmin`) IAM role on your project. For more information about granting roles, see [Manage access to projects, folders, and organizations](https://cloud.google.com/iam/docs/granting-changing-revoking-access).

### Important: Resource Cleanup

Agent Retrieval resources incur costs when active. Make sure to run the cleanup section at the end of this tutorial to delete the data store, its backing Collection, and the Cloud Storage bucket.

## Install the Libraries

We'll call the data store and SearchDocuments APIs over REST with `google-auth` and `requests`. We'll also install the `google-cloud-vectorsearch` SDK to look at the backing Collection later.

In [ ]:
%pip install google-auth requests google-cloud-vectorsearch

Then, set `PROJECT_ID` for your Google Cloud project. Managed ingestion runs in **`us-east4`** and **`europe-west4`** only, so your data store must be in one of them. You can leave `LOCATION` as the default value.

In [ ]:
# Set PROJECT_ID and LOCATION
PROJECT_ID = "your-project-id"  # @param {type:"string"}
LOCATION = "us-east4"  # @param ["us-east4", "europe-west4"]

# Validate PROJECT_ID is set
if PROJECT_ID == "your-project-id" or not PROJECT_ID or PROJECT_ID == "":
    raise ValueError(
        "⚠️ Please set PROJECT_ID to your actual Google Cloud project ID in the cell above"
    )

print(f"✅ Using project: {PROJECT_ID}")
print(f"✅ Using location: {LOCATION}")

## Authentication

On Colab, run the following to authenticate calls to the Agent Retrieval APIs. For Colab Enterprise and Cloud Workbench, you can skip this part.

In [ ]:
import sys

# Additional authentication is required for Google Colab
if "google.colab" in sys.modules:
    # Authenticate user to Google Cloud
    from google.colab import auth

    auth.authenticate_user()

## Enable APIs

Run the following command to enable the APIs used in this notebook:

- `vectorsearch.googleapis.com`: Agent Retrieval (data stores, Collections, and search)
- `aiplatform.googleapis.com`: The embedding model that embeds your documents and queries
- `discoveryengine.googleapis.com`: The ranking API used by the reranking option

In [ ]:
! gcloud services enable vectorsearch.googleapis.com aiplatform.googleapis.com discoveryengine.googleapis.com --project "{PROJECT_ID}"

## Grant the Agent Retrieval Service Agent Access

Managed ingestion reads your files and writes error reports with the Agent Retrieval **service agent**, `service-{PROJECT_NUMBER}@gcp-sa-vectorsearch.iam.gserviceaccount.com`. The following cell makes sure the service agent exists and grants it the `roles/vectorsearch.serviceAgent` role on your project.

In [ ]:
PROJECT_NUMBER = !gcloud projects describe {PROJECT_ID} --format="value(projectNumber)"
PROJECT_NUMBER = PROJECT_NUMBER[0]
SERVICE_AGENT = f"service-{PROJECT_NUMBER}@gcp-sa-vectorsearch.iam.gserviceaccount.com"

! gcloud beta services identity create --service=vectorsearch.googleapis.com --project={PROJECT_ID}
! gcloud projects add-iam-policy-binding {PROJECT_ID} --member="serviceAccount:{SERVICE_AGENT}" --role="roles/vectorsearch.serviceAgent" --condition=None --quiet > /dev/null

print(f"✅ Granted roles/vectorsearch.serviceAgent to {SERVICE_AGENT}")

-----

# Part 1: Stage Your Documents in Cloud Storage

Managed ingestion reads documents from Cloud Storage. There is no parser to choose and no chunking code to write: **one Cloud Storage object becomes one document**.

**What you'll accomplish:**
- Create a Cloud Storage bucket for the import manifest and error reports
- Write a JSONL manifest that lists each document with its ID and metadata

## Two Ways to Point an Import at Your Files

1. **URI patterns**: Pass patterns such as `gs://my-bucket/reports/*.pdf`. Each matched file becomes a document, and its ID is derived from its path. Wildcards don't recurse into subfolders.
2. **JSONL manifest**: Pass one JSONL file that lists each document on its own line, with the document ID you choose, the file's `gcsUri`, and optional metadata. **Use this when you want metadata to filter on or return in search.**

We'll use a JSONL manifest so that every document carries `doc_type`, `year`, and `quarter`. The PDFs themselves stay in the public `cloud-samples-data` bucket; only the manifest and the error reports go into your own bucket.

## Create a Cloud Storage Bucket

First, let's pick IDs for the data store and the bucket. Data store IDs are 1 to 63 characters of lowercase letters, digits, and hyphens.

In [ ]:
from datetime import datetime

data_store_id = f"alphabet-reports-{datetime.now().strftime('%m%d%y-%H%M%S')}"
BUCKET_NAME = f"{PROJECT_ID}-{data_store_id}"

print(f"Data store ID: {data_store_id}")
print(f"Bucket: gs://{BUCKET_NAME}")

In [ ]:
! gcloud storage buckets create gs://{BUCKET_NAME} --location={LOCATION} --project={PROJECT_ID}

## Write the JSONL Manifest

Each line of the manifest is a document:

- `name`: The full resource name of the document. **The last part becomes the document ID**, so you choose the IDs.
- `content.gcsUri`: The Cloud Storage URI of the file.
- `metadata.value`: The document's metadata. It must match the metadata schema we declare when we create the data store in Part 2.

Annual reports don't get a `quarter` property. Metadata properties are optional, so that's fine; we'll see how missing properties behave in search later.

In [ ]:
import json

SOURCE_PREFIX = "gs://cloud-samples-data/gen-app-builder/search/alphabet-investor-pdfs"

# Document ID -> file name. The ID encodes the year, and the quarter for earnings releases.
documents = {
    "2021-q1-earnings-release": "2021Q1_alphabet_earnings_release.pdf",
    "2021-q2-earnings-release": "2021Q2_alphabet_earnings_release.pdf",
    "2021-q3-earnings-release": "2021Q3_alphabet_earnings_release.pdf",
    "2021-q4-earnings-release": "2021Q4_alphabet_earnings_release.pdf",
    "2022-q1-earnings-release": "2022Q1_alphabet_earnings_release.pdf",
    "2022-q2-earnings-release": "2022Q2_alphabet_earnings_release.pdf",
    "2022-q3-earnings-release": "2022Q3_alphabet_earnings_release.pdf",
    "2022-q4-earnings-release": "2022Q4_alphabet_earnings_release.pdf",
    "2023-q1-earnings-release": "2023q1-alphabet-earnings-release.pdf",
    "2023-q2-earnings-release": "2023q2-alphabet-earnings-release.pdf",
    "2023-q3-earnings-release": "2023q3-alphabet-earnings-release.pdf",
    "2023-q4-earnings-release": "2023q4-alphabet-earnings-release.pdf",
    "2024-q1-earnings-release": "2024q1-alphabet-earnings-release-pdf.pdf",
    "2024-q2-earnings-release": "2024q2-alphabet-earnings-release.pdf",
    "2024-q3-earnings-release": "2024q3-alphabet-earnings-release.pdf",
    "2024-q4-earnings-release": "2024q4-alphabet-earnings-release.pdf",
    "2021-annual-report": "2021_alphabet_annual_report.pdf",
    "2022-annual-report": "2022_alphabet_annual_report.pdf",
}

with open("docs.jsonl", "w") as f:
    for doc_id, file_name in documents.items():
        year = int(doc_id[:4])
        if "earnings-release" in doc_id:
            metadata = {
                "doc_type": "earnings_release",
                "year": year,
                "quarter": int(doc_id[6]),
            }
        else:
            metadata = {"doc_type": "annual_report", "year": year}  # No quarter
        line = {
            "name": f"projects/{PROJECT_ID}/locations/{LOCATION}/dataStores/{data_store_id}/documents/{doc_id}",
            "content": {"gcsUri": f"{SOURCE_PREFIX}/{file_name}"},
            "metadata": {"value": metadata},
        }
        f.write(json.dumps(line) + "\n")

print(f"✅ Wrote {len(documents)} documents to docs.jsonl")
! head -n 2 docs.jsonl

Upload the manifest to your bucket:

In [ ]:
MANIFEST_URI = f"gs://{BUCKET_NAME}/manifests/docs.jsonl"
! gcloud storage cp docs.jsonl {MANIFEST_URI}

-----

# Part 2: Create a Data Store and Import Documents

Now that the manifest is in place, let's create a **data store** and import the documents into it.

**What you'll accomplish:**
- Set up an authorized REST session for the `v1alpha` API
- Choose how documents are parsed and segmented
- Declare a metadata schema
- Create the data store and inspect the Collection it creates
- Import the documents and list what was ingested

## Set Up the REST Session

The data store and SearchDocuments APIs are not yet in the Python SDK, so we call them with an `AuthorizedSession` from `google-auth`, which attaches your credentials to every request. We also define two small helpers:

- `check()`: Raises an error with the API's message when a call fails
- `wait()`: Polls a long-running operation (LRO) until it finishes

In [ ]:
import time

import google.auth
import requests
from google.auth.transport.requests import AuthorizedSession

credentials, _ = google.auth.default(
    scopes=["https://www.googleapis.com/auth/cloud-platform"]
)
session = AuthorizedSession(credentials)

API_BASE = "https://vectorsearch.googleapis.com/v1alpha"
PARENT_URL = f"{API_BASE}/projects/{PROJECT_ID}/locations/{LOCATION}"
DATA_STORE_URL = f"{PARENT_URL}/dataStores/{data_store_id}"


def check(response: requests.Response) -> dict:
    """Return the JSON body, or raise an error with the API's message."""
    if not response.ok:
        raise RuntimeError(f"{response.status_code}: {response.text}")
    return response.json()


def wait(operation: dict, poll_seconds: int = 10) -> dict:
    """Poll a long-running operation until it finishes, then return it."""
    start = time.time()
    while True:
        op = check(session.get(f"{API_BASE}/{operation['name']}"))
        if op.get("done"):
            if "error" in op:
                raise RuntimeError(op["error"])
            return op
        elapsed = int(time.time() - start)
        print(f"   Still running... ({elapsed // 60} min {elapsed % 60} sec elapsed)")
        time.sleep(poll_seconds)

## What is a Data Store?

A **data store** holds your documents and the **processing configuration** that controls how they are parsed, segmented, and embedded. Only `document_type` is required; everything else has a default.

### Processing Options for Documents

| Setting | Options | Default |
|---|---|---|
| **Parsing** (`parsing_config`) | `digital_parsing_config`: extracts text<br>`layout_parsing_config`: also detects tables, headings, and images | Digital parsing |
| **Segmentation** (`segmentation_config`) | `chunk_segmentation_config`: chunks of up to `chunk_size` tokens (300-500), optionally with `include_ancestor_headings`<br>`page_segmentation_config`: one segment per page | Chunk segmentation, `chunk_size` 500 tokens |
| **Embedding model** | Not settable | `gemini-embedding-2`, 768 dimensions |

Two rules to keep in mind:

- If you include a `parsing_config` or `segmentation_config` block, you must pick a mode inside it. An empty block is rejected; leave the block out to get the default.
- **Page segmentation requires digital parsing.** It is also the only mode that records **page numbers**, which you need to cite pages or filter by page range.

For this demo we use **page segmentation with digital parsing**, so every search result tells us which page of which PDF it came from. For long-form documents where a page holds several unrelated topics, chunk segmentation with layout parsing often gives more focused segments.

### Metadata Schema

To filter on or return per-document metadata, declare it in `document_metadata_schema` as a [JSON Schema](https://json-schema.org/). Supported types are `string`, `integer`, `number`, `boolean`, `array`, and `object`. You can add properties later, but you can't remove or retype them.

## Create the Data Store

Let's create the data store with our processing configuration and metadata schema. This takes a few seconds.

In [ ]:
# Create a data store for unstructured documents
create_op = check(
    session.post(
        f"{PARENT_URL}/dataStores",
        params={"data_store_id": data_store_id},
        json={
            "display_name": "Alphabet investor reports",
            "document_type": "UNSTRUCTURED",  # UNSTRUCTURED or VIDEO. Immutable.
            "processing_config": {
                "unstructured_processing_config": {
                    # One segment per page, so results carry page numbers
                    "segmentation_config": {"page_segmentation_config": {}},
                    # Page segmentation requires digital parsing
                    "parsing_config": {"digital_parsing_config": {}},
                }
            },
            # Metadata properties we can filter on and return in search
            "document_metadata_schema": {
                "schema": {
                    "type": "object",
                    "properties": {
                        "doc_type": {"type": "string"},
                        "year": {"type": "integer"},
                        "quarter": {"type": "integer"},
                    },
                }
            },
        },
    )
)

print(f"⏳ Creating data store '{data_store_id}'...")
wait(create_op)
print("✅ Data store created!")

### Key Points:

What we just accomplished:

✅ **Created a data store** for unstructured documents  
✅ **Configured page segmentation with digital parsing**, so each segment is one page with its page number  
✅ **Declared a metadata schema** with three properties: `doc_type`, `year`, `quarter`  
✅ **Got a backing Collection for free**: no `CreateCollection` call, no data or vector schema to design

## Inspecting the Data Store

Let's check the data store. Its `state` should be `ACTIVE`, and `collectionName` tells you which Collection was created for it.

In [ ]:
data_store = check(session.get(DATA_STORE_URL))

print(f"Name:            {data_store['name']}")
print(f"State:           {data_store['state']}")
print(f"Collection name: {data_store['collectionName']}")
print("Processing config:")
print(json.dumps(data_store["processingConfig"], indent=2))

The Collection name is `google-` plus your data store ID (or a hash, if the result would be too long). The `google-` prefix marks it as managed for you. It is a regular Agent Retrieval Collection, and we'll query it directly at the end of this notebook.

## Import the Documents

Now let's import the documents listed in the manifest. Two settings matter here:

- `source_format: SOURCE_FORMAT_DOCUMENT_JSON` tells the import that the URI is a JSONL manifest, not a content file. Wildcards don't apply in this mode, so name the manifest file exactly.
- `import_output_config.error_gcs_prefix` is where per-file failures are written. It's optional, but without it you have nowhere to look when a file fails. **It must point to an empty Cloud Storage folder**, so use a new prefix for each import.

In [ ]:
import_op = check(
    session.post(
        f"{DATA_STORE_URL}:importDocuments",
        json={
            "gcs_source": {
                "input_uris": [MANIFEST_URI],
                "source_format": "SOURCE_FORMAT_DOCUMENT_JSON",
            },
            "import_output_config": {
                "error_gcs_prefix": f"gs://{BUCKET_NAME}/import-errors/{datetime.now().strftime('%Y%m%d-%H%M%S')}/"
            },
        },
    )
)

print("✅ Import started")
print(f"   LRO: {import_op['name']}")

### Waiting for the Import

The import runs server-side as a long-running operation. For these 18 PDFs it typically takes **around 8-10 minutes**. While it runs:

- **Watch the operation, not the bucket.** Error files are copied into your error prefix only after the whole operation completes.
- **One import at a time.** Starting another import on the same data store while one is running returns `409 ABORTED`.

If you hit Colab's cell execution timeout, simply re-run the cell to continue waiting.

In [ ]:
print("⏳ Waiting for the import to finish...")
import_result = wait(import_op, poll_seconds=30)

print(
    f"✅ Import complete! {import_result['metadata'].get('successCount', 0)} documents imported"
)

### Listing the Documents

Let's list the documents in the data store to confirm they made it in, along with their metadata. Each document has a `state` of `SUCCEEDED` once it has been processed.

In [ ]:
docs = check(session.get(f"{DATA_STORE_URL}/documents", params={"pageSize": 100}))

for doc in sorted(docs["documents"], key=lambda d: d["name"]):
    doc_id = doc["name"].split("/")[-1]
    print(f"{doc_id:26} {doc['state']:10} {doc.get('metadata', {}).get('value')}")

### Re-running an Import

Imports are idempotent: if you import the same files again, existing documents are updated, not duplicated, and files whose content didn't change are skipped. To remove documents, use the `:purgeDocuments` method of the data store.

-----

# Part 3: Search with SearchDocuments

With the documents ingested, let's search them! This is the core of this notebook.

**What you'll accomplish:**
- Run a simple text search against the data store
- Understand the result structure: segments, snippets, page ranges, and source URIs

## Your First Search

A SearchDocuments request needs only two things:

- **The data store** as the parent: `.../dataStores/{data_store_id}/documents:search`
- **A text query**: `text_query.text`

`top_k` (default 10, max 100) sets the number of results. That's it: no Collection name, no vector field, no task type, no output fields.

**Scenario**: An analyst asks "How much did Google Cloud revenue grow?"

In [ ]:
SEARCH_URL = f"{DATA_STORE_URL}/documents:search"

response = check(
    session.post(
        SEARCH_URL,
        json={
            "text_query": {"text": "How much did Google Cloud revenue grow?"},
            "top_k": 5,
        },
    )
)

# Look at the structure of the first result
first = response["results"][0]
first["unstructuredDetails"]["contentSnippet"] = (
    first["unstructuredDetails"]["contentSnippet"][:300] + "..."
)
print(json.dumps(first, indent=2))

### Understanding the Results

Each result is a **segment** of a document, not a whole document:

- `document.name`: The document the segment came from. Several results can point to the same document.
- `document.content.gcsUri`: The original file in Cloud Storage, for citation.
- `unstructuredDetails.contentSnippet`: The full text of the segment (here, the whole page).
- `unstructuredDetails.pageRange`: The page span in the source file (1-based, inclusive). With page segmentation, `startPage` and `endPage` are the same page. With chunk segmentation, there is no page range.

Results come back in relevance order. Let's write a small helper to print them compactly: the document ID, the page, and the beginning of the segment. With the optional `keyword` argument, it shows the first line of the segment that contains the keyword instead.

In [ ]:
def print_results(response: dict, keyword: str | None = None, width: int = 110) -> None:
    """Print SearchDocuments results, one segment per entry."""
    for i, result in enumerate(response.get("results", []), 1):
        document = result["document"]
        details = result.get("unstructuredDetails", {})
        doc_id = document["name"].split("/")[-1]
        page = details.get("pageRange", {}).get("startPage", "-")
        metadata = document.get("metadata", {}).get("value")

        text = details.get("contentSnippet", "")
        if keyword:
            lines = [
                line for line in text.splitlines() if keyword.lower() in line.lower()
            ]
            text = lines[0] if lines else f"(no mention of '{keyword}')"
        text = " ".join(text.split())

        print(f"{i:2}. {doc_id}, page {page}" + (f"  {metadata}" if metadata else ""))
        print(f"    {text[:width]}")


print_results(response)

The results include the management's discussion and analysis (MD&A) pages of the annual report and the segment results tables of the quarterly earnings releases, which all report Google Cloud revenue. Behind this single call, SearchDocuments ran a semantic search and a full-text search over the backing Collection and fused the two result lists.

-----

# Part 4: Filter by Metadata and Page Range

Often you know more than the query text: the year you care about, the type of document, or where in the document the answer is likely to be. Pass `filter` to search only the segments that match.

## Filter Paths

The filter is a JSON object over the paths a result carries, so you filter on the same names you read back:

| Path | Data store type | Notes |
|---|---|---|
| `metadata.<property>` | Any | Must be declared in the data store's `document_metadata_schema`. Use dotted paths for nested properties, e.g. `metadata.issuer.country`. |
| `unstructured_details.page_range.start_page` / `end_page` | `UNSTRUCTURED` | Populated only with page segmentation. |
| `video_details.start_offset` / `end_offset` | `VIDEO` | In seconds. Fractions are allowed. |

## Filter Operators

The syntax is the same as the [Agent Retrieval query language](https://docs.cloud.google.com/gemini-enterprise-agent-platform/build/vector-search-2/query-search/query#filter-syntax):

**Comparison operators**: `$eq`, `$ne`, `$lt`, `$lte`, `$gt`, `$gte`  
**Array operators**: `$in`, `$nin`, `$all`  
**Existence**: `$exists`  
**Logical operators**: `$and`, `$or`, `$not` (nested at most two levels deep)

Let's see some examples.

### Example 1: Filter by Metadata

**Scenario**: The analyst only wants recent numbers, so we search **earnings releases from 2023 onward**.

In [ ]:
response = check(
    session.post(
        SEARCH_URL,
        json={
            "text_query": {"text": "How much did Google Cloud revenue grow?"},
            "top_k": 5,
            "filter": {
                "$and": [
                    {"metadata.doc_type": {"$eq": "earnings_release"}},
                    {"metadata.year": {"$gte": 2023}},
                ]
            },
        },
    )
)

print("Earnings releases from 2023 onward:")
print_results(response)

### Example 2: Filter by Page Range

**Scenario**: The first page of each earnings release is the press release headline, with quotes from the CEO and CFO. To find what the CEO said about AI, we search **only page 1** of each document.

In [ ]:
response = check(
    session.post(
        SEARCH_URL,
        json={
            "text_query": {"text": "What did the CEO say about AI?"},
            "top_k": 5,
            "filter": {"unstructured_details.page_range.start_page": {"$eq": 1}},
        },
    )
)

print("Page 1 only:")
print_results(response, keyword="Sundar Pichai")

### Filtering Tips

- **Write every condition with an operator.** Use `{"metadata.year": {"$eq": 2023}}`, not `{"metadata.year": 2023}`; implicit equality returns `INVALID_ARGUMENT`.
- **Page ranges need page segmentation.** Chunk segmentation doesn't record page numbers, so a page-range filter on a chunked data store matches nothing and returns no error.
- **Missing properties:** `$ne` and `$not` also match documents that don't have the property. In our data, `{"metadata.quarter": {"$ne": 4}}` also matches the annual reports, which have no `quarter`. Add `{"metadata.quarter": {"$exists": true}}` to exclude them.
- **`$in` and `$nin` take strings.** A numeric list returns `INVALID_ARGUMENT`. For numbers, use a range or an `$or` of `$eq` conditions.
- **Undeclared properties are rejected.** Filtering on a property that isn't in the metadata schema returns `INVALID_ARGUMENT`.

-----

# Part 5: Return Document Metadata

By default, results carry no metadata. To get metadata back on each result, set `output_fields.metadata` to the properties you want, named as declared in the schema (without the `metadata.` prefix the filter uses). Pass `["*"]` to return every declared property.

**Scenario**: Show the year and quarter of each matching page, so the analyst can see at a glance which period each answer is from.

In [ ]:
response = check(
    session.post(
        SEARCH_URL,
        json={
            "text_query": {"text": "How many employees does Alphabet have?"},
            "top_k": 5,
            "output_fields": {"metadata": ["*"]},  # Or name them: ["year", "quarter"]
        },
    )
)

print_results(response, keyword="employees")

Each result now carries `document.metadata.value` with the requested properties. A property the document has no value for is left out: the annual report has no `quarter`, so its metadata shows only `doc_type` and `year`. `output_fields` works together with `filter`, `document_ids`, and reranking.

Notice the ranking, too. The query asks for a headcount, but some of the top results are balance sheet pages that don't mention employees at all. We'll fix that with reranking in Part 7.

-----

# Part 6: Restrict the Search to Specific Documents

When you already know which documents are relevant, pass `document_ids` to search only within them. You can pass up to 100 IDs. `document_ids` and `filter` combine with AND.

**Scenario**: Compare operating income in the Q4 2023 and Q4 2024 earnings releases.

In [ ]:
response = check(
    session.post(
        SEARCH_URL,
        json={
            "text_query": {"text": "Total operating income"},
            "top_k": 4,
            "document_ids": ["2023-q4-earnings-release", "2024-q4-earnings-release"],
        },
    )
)

print_results(response)

All results come from the two documents we named. We chose the document IDs ourselves in the manifest, which makes this easy; to look them up, list the data store's documents as we did in Part 2.

-----

# Part 7: Improve Ranking with Reranking

By default, SearchDocuments returns results in the fused order of the semantic and full-text searches. Turn on **reranking** to run a second pass with the Vertex AI ranking API, which scores each candidate segment against the query. This improves relevance at the cost of some added latency.

**Scenario**: "How many employees does Alphabet have?" Let's run the query with and without reranking, and show the line of each page that mentions employees.

In [ ]:
query = "How many employees does Alphabet have?"

for reranking in [False, True]:
    response = check(
        session.post(
            SEARCH_URL,
            json={
                "text_query": {"text": query},
                "top_k": 5,
                "ranking_spec": {"reranking_spec": {"enabled": reranking}},
            },
        )
    )
    print(f"Reranking {'ON' if reranking else 'OFF'}:")
    print_results(response, keyword="employees")
    print()

With reranking on, every top result is a page that actually reports the headcount: the annual report's "Alphabet had 190,234 employees" and the "Number of employees" row of the quarterly supplemental information tables. The balance sheet pages, which don't mention employees at all, drop out.

### How Reranking Works

When reranking is enabled, SearchDocuments retrieves more candidates than `top_k`, reranks them, then returns the top `top_k`. `rank_oversampling_factor` (default 5) controls how many candidates are retrieved, as a multiple of `top_k`:

```python
"ranking_spec": {
    "reranking_spec": {
        "enabled": True,
        "rank_oversampling_factor": 3,  # 1 = rerank only the initial top_k results
    }
}
```

**Things to know:**
- **Prerequisite**: Reranking requires the Discovery Engine API (`discoveryengine.googleapis.com`), which we enabled in the setup section. Without it, the call returns `FAILED_PRECONDITION`.
- **Graceful degradation**: If the ranking service is temporarily unavailable, the call still succeeds and returns results in the fused order, with a warning that reranking was skipped.

-----

# Part 8: Under the Hood: The Backing Collection

SearchDocuments is a thin layer over the Collection that managed ingestion created. To see what it saves you, let's run a comparable search directly on that Collection with the `google-cloud-vectorsearch` SDK, as you would in the [Introduction to Vector Search 2.0](https://github.com/GoogleCloudPlatform/generative-ai/blob/main/embeddings/vector-search-2-intro.ipynb) notebook.

To query the Collection directly, you need to know:

- **The Collection name**, from the data store's `collectionName` (not the data store name)
- **The vector field**, `multimodal_embedding`, and the task type `RETRIEVAL_QUERY`
- **The stored field names**: fields the service writes are prefixed with `google_` (e.g. `google_document_id`, `google_customer_uri`, `google_unstructured_details.content_snippet`), and your metadata properties keep their declared names (`year`, `quarter`)
- **`output_fields`**: without it, you get back only the object name and distance

The server embeds your query text with your credentials, so this call also needs the `roles/aiplatform.user` role on the project. Without it, the search fails with `403 PERMISSION_DENIED` on `aiplatform.endpoints.predict`.

In [ ]:
from google.cloud import vectorsearch_v1

data_object_search_service_client = vectorsearch_v1.DataObjectSearchServiceClient()

request = vectorsearch_v1.SearchDataObjectsRequest(
    parent=data_store["collectionName"],  # The backing Collection, not the data store
    semantic_search=vectorsearch_v1.SemanticSearch(
        search_text="How much did Google Cloud revenue grow?",
        search_field="multimodal_embedding",
        task_type=vectorsearch_v1.EmbeddingTaskType.RETRIEVAL_QUERY,
        top_k=3,
        filter={"year": {"$gte": 2023}},  # Stored name, not metadata.year
        output_fields=vectorsearch_v1.OutputFields(
            data_fields=[
                "google_document_id",
                "google_customer_uri",
                "google_unstructured_details.start_page",
                "year",
                "quarter",
            ]
        ),
    ),
)

for result in data_object_search_service_client.search_data_objects(request=request):
    data = result.data_object.data
    print(
        f"{data['google_document_id']}, page {data['google_unstructured_details']['start_page']:.0f} "
        f"(year={data['year']:.0f}, quarter={data['quarter']:.0f}, distance={result.distance:.3f})"
    )
    print(f"    {data['google_customer_uri'][0]}")

This is a semantic-only search. To match what SearchDocuments does, you would also wire a text search into `BatchSearchDataObjects` with RRF, and call the ranking API for reranking. Querying the Collection directly is the right choice when you need that level of control, for example to create an ANN index with your own `filter_fields` for large data stores. For everything else, SearchDocuments does it in one call.

-----

# Part 9: Clean Up

## Important: Delete Resources to Avoid Costs

Agent Retrieval resources incur costs when active. To avoid unexpected charges, delete:

1. **The data store**, its documents, and its backing Collection
2. **The Cloud Storage bucket** with the manifest and error reports

## Step 1: Delete the Data Store

A data store that still holds documents can be deleted only with `force=true`. Add `backing_collection_deleted=true` to also delete the backing Collection and its Data Objects. Without it, the `google-{data_store_id}` Collection is left behind and keeps incurring costs.

Deletion runs in the background and takes around 20 minutes. You don't need to wait for it: the data store shows the `DELETING` state while it runs, and returns `404 NOT_FOUND` once it's gone.

In [ ]:
delete_op = check(
    session.delete(
        DATA_STORE_URL,
        params={"force": "true", "backing_collection_deleted": "true"},
    )
)

print(f"🗑️ Deleting data store '{data_store_id}' and its backing Collection...")
print(f"   LRO: {delete_op['name']}")
print(f"   State: {check(session.get(DATA_STORE_URL))['state']}")

## Step 2: Delete the Cloud Storage Bucket

The source PDFs stay in the public bucket; this deletes only your bucket with the manifest and error reports.

In [ ]:
! gcloud storage rm --recursive gs://{BUCKET_NAME}
print(f"🗑️ Deleted bucket: gs://{BUCKET_NAME}")
print(
    "\n✅ Cleanup complete! The data store and its backing Collection finish deleting in the background."
)

-----

# Summary

Congratulations! You've completed an introduction to **managed ingestion and SearchDocuments in Agent Retrieval**. Let's recap what you've learned.

## What You've Learned

### 1. **Managed Ingestion** - From Files to a Searchable Collection
- Staged documents with a JSONL manifest that sets document IDs and metadata
- Created a data store with page segmentation, digital parsing, and a metadata schema
- Imported PDFs with a single long-running operation; the service parsed, segmented, and embedded them
- Got a backing Collection created for you, with no schema to design

### 2. **SearchDocuments** - One Call for Document Search
- Searched with just a data store and a text query; hybrid search runs automatically
- Read results as segments with snippets, page ranges, and source URIs for citation

### 3. **Narrowing and Enriching Results**
- Filtered on metadata (`metadata.year`) and page ranges (`unstructured_details.page_range.start_page`)
- Returned document metadata with `output_fields.metadata`
- Restricted the search to specific documents with `document_ids`

### 4. **Better Ranking**
- Turned on reranking with one toggle and saw the most relevant pages move to the top

### 5. **Under the Hood**
- Queried the backing Collection directly with the SDK, using the stored `google_` field names

## Next Steps

- [Introduction to Agent Platform Vector Search 2.0](https://github.com/GoogleCloudPlatform/generative-ai/blob/main/embeddings/vector-search-2-intro.ipynb): Learn the Collection, Data Object, search, and ANN index APIs that sit under SearchDocuments
- [Agent Retrieval documentation](https://docs.cloud.google.com/gemini-enterprise-agent-platform/build/vector-search-2/overview)
- Try your own files: put PDFs, Word, PowerPoint, Excel, or text files in a bucket and point an import at `gs://your-bucket/your-prefix/*`

-----

# Appendix: SearchDocuments Reference

## Request Fields

| Field | Required | Default | Notes |
|---|---|---|---|
| Data store name | Yes | | In the URL: `projects/{project}/locations/{location}/dataStores/{data_store}/documents:search` |
| `text_query.text` | Yes | | The search text |
| `top_k` | No | 10 | Number of results. Values above 100 are coerced to 100. |
| `document_ids` | No | Search all | Restrict to these document IDs. Max 100. |
| `filter` | No | Search all | Filter over `metadata.<property>`, the page range, or video offsets (seconds). At most two levels of nesting. Combined with `document_ids` by AND. |
| `output_fields.metadata` | No | No metadata | Metadata properties to return, as declared in the schema. Nested properties use dotted paths. `"*"` returns all. |
| `ranking_spec.reranking_spec.enabled` | No | `false` | Turn on reranking after hybrid retrieval |
| `ranking_spec.reranking_spec.rank_oversampling_factor` | No | 5 | How many candidates to retrieve before reranking, as a multiple of `top_k`. Ignored if reranking is off. |

## Response Fields

| Field | Description |
|---|---|
| `results[].document.name` | Full resource name of the document the segment came from |
| `results[].document.content.gcsUri` | The Cloud Storage URI of the original file |
| `results[].document.metadata.value` | The metadata properties named in `output_fields.metadata`. Absent when none were requested or present. |
| `results[].unstructuredDetails.contentSnippet` | The full text of the segment |
| `results[].unstructuredDetails.pageRange.startPage` / `endPage` | Page span in the source document (1-based, inclusive). Page segmentation only. |
| `results[].videoDetails.startOffset` / `endOffset` | Time span of a video segment |

Only `name`, `content.gcsUri`, and requested metadata are populated on `document`. For the full document (state, timestamps, all metadata), call `GET` on the returned `document.name`.

## Common Errors

| Condition | Code | What to do |
|---|---|---|
| Missing `vectorsearch.documents.search` permission | `PERMISSION_DENIED` | Grant the `vectorsearch.viewer` role on the data store's project |
| Data store does not exist | `NOT_FOUND` | Check the data store ID and region |
| Data store exists but nothing has been imported | `OK`, zero results | Run an import first |
| More than 100 `document_ids` | `INVALID_ARGUMENT` | Reduce the list |
| `filter` names a property the schema doesn't declare | `INVALID_ARGUMENT` | Use a declared property, written as `metadata.<property>` |
| `filter` uses a stored field name (e.g. a `google_` field) | `INVALID_ARGUMENT` | Use `metadata.<property>`, a `page_range` path, or a `video_details` offset |
| Operand of the wrong type (e.g. `{"metadata.year": {"$lt": "ten"}}`) | `INVALID_ARGUMENT` | Match the property's declared type |
| `filter` nested more than two levels | `INVALID_ARGUMENT` | Remove one level of nesting |
| Page-range filter returns no results | `OK`, zero results | The data store uses chunk segmentation; page filters need page segmentation |
| Discovery Engine API not enabled (reranking requested) | `FAILED_PRECONDITION` | `gcloud services enable discoveryengine.googleapis.com` |
| Ranking service unavailable | `OK` + warning | Results come back in fused order; reranking was skipped |